<!-- nav -->
[← 4 · Values with widths](04_Values_With_Widths.ipynb) · [Home](../../README.md) · [6 · Analyzing and rewriting rules →](06_Analyzing_And_Rewriting_Rules.ipynb)

# 5 · Deciding what you can now

**The problem.** Two questions keep coming back. A new product arrives with its own limits (a frozen vaccine at −25 to
−15 °C, for up to 120 hours) and the rule must be written again, slightly differently. And while a shipment is still on
the road, "will it arrive in time?" depends on a number nobody has yet: how many more hours the trip takes.

**Partial evaluation** answers both. Give it the variables you know, and it evaluates everything they decide, leaving
the rest as an expression: the **residual**, the question that's left.

In [1]:
from mbse.Expressions import Evaluators, Expressions as E, Partials
from mbse.Expressions.Dialects.Python import Text
from mbse.Expressions.Framework import Symbolics
from toolkit import cold_chain, show

store = cold_chain()
template = Text.FromFunction(
    lambda this, low, high, max_hours, to_go: low <= this.celsius <= high and this.hours + to_go <= max_hours)
print(show(template))
Symbolics.free(template.data)

and(and(le(low, get(this, 'celsius')), le(get(this, 'celsius'), high)), le(add(get(this, 'hours'), to_go), max_hours))


{'high', 'low', 'max_hours', 'this', 'to_go'}

`Symbolics.free` gives the names a rule needs from its scope: the shipment, the product's limits, and the hours to go.

## Step 1: one template, many products

Binding the limits gives each product its rule:

In [2]:
vaccine = Partials.OfAny(template, {"low": 2.0, "high": 8.0, "max_hours": 72})
frozen = Partials.OfAny(template, {"low": -25.0, "high": -15.0, "max_hours": 120})
for rule in (vaccine, frozen):
    print(show(rule), sorted(Symbolics.free(rule)))

and(and(le(2.0, get(this, 'celsius')), le(get(this, 'celsius'), 8.0)), le(add(get(this, 'hours'), to_go), 72)) ['this', 'to_go']
and(and(le(-25.0, get(this, 'celsius')), le(get(this, 'celsius'), -15.0)), le(add(get(this, 'hours'), to_go), 120)) ['this', 'to_go']


The residuals are ordinary expressions: they save, translate and evaluate like any other, and don't refer back to
the template.

## Step 2: what the shipment decides

Now bind a shipment, but not the hours to go. Whatever the shipment decides is evaluated, and Kleene's logic (case
study 2) simplifies what's left: `and(true, x)` is `x`, and `and(false, x)` is false whatever `x` is.

In [3]:
s1 = store.Shipment().id("S1").celsius(5.5).hours(30).create()
s2 = store.Shipment().id("S2").celsius(9.1).hours(30).create()
s3 = store.Shipment().id("S3").hours(20).create()  # no temperature yet
for shipment in (s1, s2, s3):
    print(shipment.id, show(Partials.OfAny(vaccine, {"this": shipment})))

S1 le(add(30, to_go), 72)
S2 False
S3 and(and(le(2.0, get(this, 'celsius')), le(get(this, 'celsius'), 8.0)), le(add(20, to_go), 72))


- S1 is in range, so all that's left is the clock: it arrives in time if 30 more hours, plus what's to go, stay
  within 72.
- S2 is out of range, so it fails whatever the trip takes.
- S3 has no temperature, so its range check stays as it was, still about `this`, and the clock check is reduced.

## Step 3: the residual agrees with the rule

A residual is only useful if it means the same thing. Finishing it with the rest of the variables gives what the
original gives with all of them:

In [4]:
for shipment in (s1, s2, s3):
    residual = Partials.OfAny(vaccine, {"this": shipment})
    for to_go in (10, 50):
        whole = Evaluators.OfAny(vaccine, {"this": shipment, "to_go": to_go})
        assert Evaluators.OfAny(residual, {"this": shipment, "to_go": to_go}) == whole
        print(shipment.id, to_go, whole)

S1 10 True
S1 50 False
S2 10 False
S2 50 False
S3 10 None
S3 50 None


## Step 4: lets of known values

A let whose value is known binds it in its body, and disappears once nothing refers to it:

In [5]:
total = Text.FromFunction(lambda this, to_go: (lambda hours: hours <= 72)(this.hours + to_go))
print(show(total))
print(show(Partials.OfAny(total, {"this": s1})))
print(show(Partials.OfAny(total, {"this": s1, "to_go": 50})))

let hours = add(get(this, 'hours'), to_go) in le(hours, 72)
let hours = add(30, to_go) in le(hours, 72)
False


With `to_go` unknown, the let stays, now binding `add(30, to_go)`. With both known, the whole rule is a value.

## Why it works this way

- **The residual is the question that's left.** Hand it to whoever knows the rest, such as the routing service that
  knows the hours to go, and they can answer it without knowing anything about shipments.
- **Specializing is evaluating.** A template's parameters are variables like any other, so specializing it needs no
  new mechanism, and the specialized rule is checked and stored like any rule.
- **Closed parts are evaluated whole.** A subexpression whose variables are all known is evaluated, and replaced by its
  value when it has one. When its value is unknown, as S3's temperature is, it stays as written: the residual still
  says what to ask.

<!-- nav -->
[← 4 · Values with widths](04_Values_With_Widths.ipynb) · [Home](../../README.md) · [6 · Analyzing and rewriting rules →](06_Analyzing_And_Rewriting_Rules.ipynb)